# Generador de nombres de dinosaurios — entrenamiento y evaluación

Este notebook solo orquesta: la lógica vive en `labdino` (`preprocessing`, `model`, `training`, `reporting`).

**Diseño experimental (fijado antes de entrenar):**
- Dos configuraciones que solo difieren en capacidad: A = GRU 1 capa / hidden 128; B = GRU 2 capas / hidden 256 con dropout 0.2 entre capas.
- Mismo split, semilla, batch size, optimizador (Adam) y learning rate.
- Máximo de épocas fijo con early stopping por paciencia sobre la pérdida de validación; se conserva el mejor checkpoint.
- **Criterio de elección: menor pérdida de validación.**

In [ ]:
from pathlib import Path

import torch

from labdino.model import CharRNN, ModelConfig
from labdino.preprocessing import Vocab, build_dataset, load_names
from labdino.reporting import save_curves, save_summary
from labdino.seed import set_seed
from labdino.training import TrainConfig, split_train_val, train_model

ROOT = Path.cwd().parent if Path.cwd().name == "name_generator" else Path.cwd()
EVIDENCE = ROOT / "evidence"
ARTIFACTS = ROOT / "name_generator" / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)  # CPU, reproducible

In [ ]:
SEED = 42
names = load_names(ROOT / "data" / "dinos.csv")
vocab = Vocab.from_names(names)
X, Y, T = build_dataset(names, vocab)
train, val = split_train_val(X, Y, val_fraction=0.1, seed=SEED)
print(f"{len(names)} nombres | |V|={len(vocab)} | T={T} | train={train[0].size(0)} val={val[0].size(0)}")

In [ ]:
train_cfg = TrainConfig(batch_size=64, lr=3e-3, max_epochs=100, patience=10, seed=SEED)
model_cfgs = {
    "A": ModelConfig(len(vocab), cell="gru", embedding_dim=32, hidden_size=128, num_layers=1, dropout=0.0, pad_id=vocab.pad_id),
    "B": ModelConfig(len(vocab), cell="gru", embedding_dim=32, hidden_size=256, num_layers=2, dropout=0.2, pad_id=vocab.pad_id),
}
print(train_cfg)

In [ ]:
results, rows = {}, []
for name, mcfg in model_cfgs.items():
    set_seed(SEED)
    model = CharRNN(mcfg)
    res = train_model(model, train, val, vocab.pad_id, train_cfg)
    results[name] = res
    torch.save({"state_dict": res.best_state, "model_config": mcfg.to_dict(), "best_epoch": res.best_epoch},
               ARTIFACTS / f"checkpoint_{name}.pt")
    rows.append({
        "config": name, "architecture": mcfg.cell.upper(), "num_layers": mcfg.num_layers,
        "hidden_size": mcfg.hidden_size, "embedding_dim": mcfg.embedding_dim, "dropout": mcfg.dropout,
        "epochs_run": res.epochs_run, "max_epochs": train_cfg.max_epochs, "patience": train_cfg.patience,
        "loss": "CrossEntropyLoss(ignore_index=<PAD>)", "optimizer": "Adam", "lr": train_cfg.lr,
        "batch_size": train_cfg.batch_size, "seed": SEED,
        "n_params": sum(p.numel() for p in model.parameters()),
        "best_val_loss": res.best_val_loss, "best_epoch": res.best_epoch,
        "final_train_loss": res.train_losses[-1],
    })
    print(f"Config {name}: épocas={res.epochs_run} mejor época={res.best_epoch} mejor val={res.best_val_loss:.4f}")

In [ ]:
CRITERION = "menor pérdida de validación"
winner = min(rows, key=lambda r: r["best_val_loss"])["config"]
save_curves(results, EVIDENCE)
save_summary(rows, winner, CRITERION, EVIDENCE)
print("Ganadora:", winner)

In [ ]:
from IPython.display import Image, Markdown, display

display(Image(EVIDENCE / "loss_curves_comparison.png"))
display(Markdown((EVIDENCE / "training_summary.md").read_text(encoding="utf-8")))

## Muestreo autoregresivo

Se usa la configuración ganadora (criterio ya fijado: menor pérdida de validación), cargando su mejor checkpoint.
Misma semilla para todas las estrategias; 1000 muestras por estrategia. Novedad contra TODO el dataset (train+val).

In [ ]:
import json
from dataclasses import asdict

from labdino.reporting import save_sampling
from labdino.sampling import SamplingConfig, evaluate_strategy, generate_names, pick_novel_names

winner = json.loads((EVIDENCE / "training_summary.json").read_text())["winner"]
ckpt = torch.load(ARTIFACTS / f"checkpoint_{winner}.pt", map_location="cpu")
best_model = CharRNN(ModelConfig(**ckpt["model_config"]))
best_model.load_state_dict(ckpt["state_dict"])
known = set(names)  # train + val, ya normalizados
print("Modelo ganador:", winner, "| mejor época:", ckpt["best_epoch"])

In [ ]:
strategies = [
    SamplingConfig("T=0.5", temperature=0.5),
    SamplingConfig("T=1.0", temperature=1.0),
    SamplingConfig("T=1.5", temperature=1.5),
    SamplingConfig("top-k=5 (T=1.0)", temperature=1.0, top_k=5),
    SamplingConfig("top-k=3 (T=1.0)", temperature=1.0, top_k=3),
    SamplingConfig("top-p=0.9 (T=1.0)", temperature=1.0, top_p=0.9),
]
N_SAMPLES = 1000
rows = []
for cfg in strategies:
    samples = generate_names(best_model, vocab, T, N_SAMPLES, cfg, torch.Generator().manual_seed(SEED))
    rows.append({**asdict(cfg), "seed": SEED, "metrics": evaluate_strategy(samples, known), "examples": samples[:5]})
    print(cfg.name, {k: round(v, 3) for k, v in rows[-1]["metrics"].items()})

In [ ]:
CHOSEN_STRATEGY = "top-p=0.9 (T=1.0)"
chosen_cfg = next(c for c in strategies if c.name == CHOSEN_STRATEGY)
generated = pick_novel_names(best_model, vocab, T, chosen_cfg, known, count=10, min_len=3, seed=123)
assert len(set(generated)) == 10 and all(n not in known and len(n) >= 3 for n in generated)
for i, n in enumerate(generated, 1):
    print(i, n.capitalize())

In [ ]:
CHOSEN_NAME = "doractodon"
ANALYSIS = """
- **Temperatura baja (0.5)** concentra la probabilidad en los caracteres más probables: el modelo reproduce con frecuencia nombres reales (7.0 % de copias) y repite mucho (14.5 % de muestras repetidas). Es poco útil para crear especies nuevas.
- **Temperatura 1.0** muestrea la distribución aprendida sin modificarla: casi no copia (1.0 %) ni repite (0.2 %), pero a veces produce nombres de menos de 3 caracteres (0.2 %).
- **Temperatura 1.5** aplana la distribución: la tasa de copias es mínima (0.1 %), pero los nombres se alargan (13.15 caracteres de media frente a ~11.6 de las demás) y aparecen más cortos degenerados (0.8 %) y secuencias menos pronunciables (por ejemplo "pocavclikamidocrechiy").
- **Top-k** recorta la cola pero deja la parte alta intacta: k=5 sube las copias a 4.4 % y los repetidos a 7.6 %, y k=3 llega a 10.7 % y 38.4 %. A menor k, menos diversidad.
- **Top-p = 0.9 (T=1.0)** elimina la cola improbable sin estrechar tanto como top-k: 1.2 % de copias, 1.1 % de repetidos, 0 % de nombres cortos y longitud media 11.65 (el dataset tiene 11.96). Frente a T=1.0 puro, que tiene la longitud media más fiel al dataset (11.97), top-p evita los nombres degenerados de menos de 3 caracteres. Por eso se eligió para generar los 10 nombres.
- Los 10 nombres reutilizan patrones reales del dataset (-saurus, -odon, -ceratops, -raptor) sin copiar ninguno. Se elige **Doractodon**: es pronunciable, de longitud moderada y su terminación -odon es de las que se usarán como ejemplo en la etapa de identidad con Ollama.
- Limitación: la novedad se mide solo por coincidencia exacta con el dataset; un nombre puede ser muy parecido a uno real (p. ej. variantes de "-saurus") y seguir contando como novedoso.
"""
save_sampling(rows, generated, CHOSEN_NAME, CHOSEN_STRATEGY, ANALYSIS, EVIDENCE)
display(Markdown((EVIDENCE / "generated_names.md").read_text(encoding="utf-8")))